# 03_silver_hl7
Parses the raw HL7 v2 messages in `health.bronze.hl7_messages` into columns, and fixes the feed's data-quality issues.

| Output table | Grain |
| --- | --- |
| `health.silver.hl7_adt_events` | one row per distinct ADT message |
| `health.silver.hl7_encounters` | one row per hospital visit (several messages roll up into one visit) |

**How HL7 v2 is structured:** a message is made of segments separated by `\r` (MSH, EVN, PID, PV1, DG1).
Inside a segment, fields are separated by `|`; inside a field, components by `^`, and repeating values by `~`.
So "PID-5.1" means: segment PID, field 5, component 1 (the family name). See `docs/hl7_fhir_field_map.md`.

**Privacy:** silver keeps only what patient matching needs (name, date of birth, sex, ZIP). SSN, street address and phone are dropped here.

In [0]:
%sql
SET TIME ZONE 'UTC';

COMMENT ON SCHEMA health.silver IS
  'Parsed, cleaned and conformed data. Keeps only the identifiers needed for patient matching (name, date of birth, sex, ZIP); SSN, street address and phone are dropped. Restricted access: gold is fully de-identified.';

## Helper: HL7 escape sequences
HL7 escapes its own delimiters inside values: `\T\` means `&`, `\F\` means `|`, and so on.
176 messages contain them (for example, facility names with an ampersand).

In [0]:
%sql
CREATE OR REPLACE TEMPORARY FUNCTION hl7_unescape(s STRING)
RETURNS STRING
RETURN replace(replace(replace(replace(replace(s,
         '\\F\\', '|'), '\\S\\', '^'), '\\T\\', '&'), '\\R\\', '~'), '\\E\\', '\\');

CREATE OR REPLACE TEMPORARY FUNCTION hl7_ts(s STRING)
RETURNS TIMESTAMP
-- The feed mixes three formats: yyyyMMddHHmmss, yyyyMMddHHmm, and yyyyMMddHHmmss+0000
RETURN coalesce(try_to_timestamp(substr(s, 1, 14), 'yyyyMMddHHmmss'),
                try_to_timestamp(substr(s, 1, 12), 'yyyyMMddHHmm'));

## Step 1: remove duplicates and split each message into its segments
Re-sent messages are byte-for-byte identical, so they share a `message_hash`. Keep the first one received.

Field numbering: in MSH, field *n* is position *n* (MSH-1 is the `|` itself); in every other segment, field *n* is position *n + 1*.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hl7_segments AS
WITH dedup AS (
  SELECT *
  FROM health.bronze.hl7_messages
  QUALIFY row_number() OVER (PARTITION BY message_hash ORDER BY ingested_at) = 1
),
segs AS (
  SELECT message_hash, source_file, ingested_at, split(raw_message, '\r') AS segments
  FROM dedup
)
SELECT
  message_hash, source_file, ingested_at,
  try_element_at(filter(segments, s -> s LIKE 'MSH|%'), 1) AS msh,
  try_element_at(filter(segments, s -> s LIKE 'PID|%'), 1) AS pid,
  try_element_at(filter(segments, s -> s LIKE 'PV1|%'), 1) AS pv1,
  filter(segments, s -> s LIKE 'DG1|%')                    AS dg1
FROM segs;

SELECT COUNT(*) AS distinct_messages FROM hl7_segments;   -- expect 3,613

## Step 2: extract the fields
`split_part(text, delimiter, n)` returns the n-th piece, or an empty string if it isn't there, so missing fields never cause errors.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.hl7_adt_events
COMMENT 'One row per distinct HL7 v2 ADT message, parsed and cleaned. SSN and street address removed.'
AS
WITH f AS (
  SELECT
    *,
    split_part(pid, '|', 4)  AS pid_3,    -- patient identifiers (repeating, separated by ~)
    split_part(pid, '|', 6)  AS pid_5,    -- name
    split_part(pid, '|', 12) AS pid_11,   -- address
    split_part(pv1, '|', 4)  AS pv1_3     -- location
  FROM hl7_segments
),
ids AS (
  SELECT
    *,
    try_element_at(filter(split(pid_3, '~'), r -> r LIKE '%^LEGACYHIS^MR'), 1) AS mrn_rep,
    try_element_at(filter(split(pid_3, '~'), r -> r LIKE '%^SYNTHEA^PI'), 1)   AS enterprise_rep
  FROM f
)
SELECT
  -- message
  split_part(msh, '|', 10)                              AS message_control_id,
  split_part(split_part(msh, '|', 9), '^', 2)           AS event_type,      -- A01 admit, A02 transfer, A03 discharge, A04 register
  split_part(msh, '|', 4)                               AS sending_facility_code,
  hl7_ts(split_part(msh, '|', 7))                       AS message_ts,

  -- patient (lpad restores MRNs that lost their leading zeros)
  lpad(split_part(mrn_rep, '^', 1), 8, '0')             AS mrn,
  nullif(split_part(enterprise_rep, '^', 1), '')        AS enterprise_id,
  hl7_unescape(split_part(pid_5, '^', 1))               AS family_name,
  hl7_unescape(split_part(pid_5, '^', 2))               AS given_name,
  lower(regexp_replace(split_part(pid_5, '^', 1), '[^A-Za-z]', '')) AS family_norm,   -- match key: no case, digits or punctuation
  lower(regexp_replace(split_part(pid_5, '^', 2), '[^A-Za-z]', '')) AS given_norm,
  to_date(try_to_timestamp(split_part(pid, '|', 8), 'yyyyMMdd'))     AS birth_date,
  nullif(split_part(pid, '|', 9), '')                   AS sex,
  hl7_unescape(split_part(pid_11, '^', 3))              AS city,
  split_part(pid_11, '^', 4)                            AS state,
  left(split_part(pid_11, '^', 5), 5)                   AS zip,

  -- visit
  split_part(pv1, '|', 20)                              AS visit_number,
  split_part(pv1, '|', 3)                               AS patient_class,   -- I inpatient, E emergency, O outpatient
  split_part(pv1_3, '^', 1)                             AS unit,
  split_part(pv1_3, '^', 4)                             AS facility_code,
  hl7_unescape(split_part(pv1_3, '^', 9))               AS facility_name,
  hl7_ts(split_part(pv1, '|', 45))                      AS admit_ts,
  hl7_ts(nullif(split_part(pv1, '|', 46), ''))          AS discharge_ts,

  -- diagnoses (DG1-3 = code^description^coding system)
  transform(dg1, d -> named_struct(
      'code',    split_part(split_part(d, '|', 4), '^', 1),
      --'display', hl7_unescape(split_part(split_part(d, '|', 4), '^', 2)),  
      'display', replace(replace(replace(replace(replace(split_part(split_part(d, '|', 4), '^', 2),
                   '\\F\\', '|'), '\\S\\', '^'), '\\T\\', '&'), '\\R\\', '~'), '\\E\\', '\\'),
      'system',  split_part(split_part(d, '|', 4), '^', 3))) AS diagnoses,

  -- lineage
  message_hash, source_file, ingested_at
FROM ids;

## Step 3: roll messages up into visits
One inpatient stay produces an A01, maybe an A02, and an A03. This collapses them into one row per visit.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.hl7_encounters
COMMENT 'One row per hospital visit from the HL7 feed: the latest patient details, admit and discharge times, and all diagnoses.'
AS
SELECT
  facility_code,
  visit_number,
  max_by(mrn, message_ts)            AS mrn,
  max(enterprise_id)                 AS enterprise_id,     -- present if ANY message for the visit carried it
  max_by(family_norm, message_ts)    AS family_norm,
  max_by(given_norm, message_ts)     AS given_norm,
  max_by(birth_date, message_ts)     AS birth_date,
  max_by(sex, message_ts)            AS sex,
  max_by(zip, message_ts)            AS zip,
  max_by(patient_class, message_ts)  AS patient_class,
  max_by(facility_name, message_ts)  AS facility_name,
  min(admit_ts)                      AS admit_ts,
  max(CASE WHEN event_type = 'A03' THEN discharge_ts END) AS discharge_ts,
  bool_or(event_type = 'A03')        AS is_discharged,
  bool_or(event_type = 'A03') AND max(CASE WHEN event_type = 'A03' THEN discharge_ts END) IS NULL
                                     AS discharge_time_missing,
  bool_or(event_type = 'A02')        AS had_transfer,
  array_distinct(flatten(collect_list(diagnoses))) AS diagnoses,
  count(*)                           AS message_count
FROM health.silver.hl7_adt_events
GROUP BY facility_code, visit_number;

## Validate

In [0]:
%sql
-- Expect A03 1,799 | A04 1,747 | A01 52 | A02 15  (3,613 total, duplicates removed)
SELECT event_type, COUNT(*) AS messages
FROM health.silver.hl7_adt_events
GROUP BY event_type ORDER BY messages DESC

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM health.silver.hl7_encounters)                          AS visits,                  -- expect 1,799
  (SELECT COUNT(DISTINCT mrn) FROM health.silver.hl7_encounters)               AS patients_by_mrn,         -- expect 88
  (SELECT COUNT(*) FROM health.silver.hl7_adt_events WHERE length(mrn) <> 8)   AS bad_mrns,                -- expect 0
  (SELECT COUNT(*) FROM health.silver.hl7_adt_events WHERE message_ts IS NULL) AS unparsed_timestamps,     -- expect 0
  (SELECT COUNT(*) FROM health.silver.hl7_encounters WHERE enterprise_id IS NULL) AS visits_without_enterprise_id,   -- expect 317 (these must be matched on demographics)
  (SELECT COUNT(*) FROM health.silver.hl7_encounters WHERE discharge_time_missing) AS discharges_missing_time,         -- expect 39
  (SELECT COUNT(*) FROM health.silver.hl7_adt_events WHERE facility_name LIKE '%&%') AS names_unescaped                  -- expect 176 (\T\ turned back into &)